# Medical Supply Inventory - EDA and Baseline ML

This notebook performs quick EDA and trains a first baseline model.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, accuracy_score, f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

In [ ]:
# Load data
data_path = Path('..') / 'data' / 'Medical_Supply_Inventory.csv'
df = pd.read_csv(data_path)

print(f'Rows, Columns: {df.shape}')
display(df.head())

In [ ]:
# Different EDA Visualizations
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load data
data_path = Path('..') / 'data' / 'Medical_Supply_Inventory.csv'
df = pd.read_csv(data_path)

# Clean Order Qty column
df['Order Qty'] = pd.to_numeric(df['Order Qty'].astype(str).str.replace(',', ''), errors='coerce')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# 1. Distribution of Order Quantity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Order Qty'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Order Qty Distribution')
axes[0].set_xlabel('Order Qty')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df['Order Qty'].dropna(), vert=True)
axes[1].set_title('Order Qty Box Plot')
axes[1].set_ylabel('Order Qty')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Top 10 Manufacturers by Product Count
top_manufacturers = df['Manufacturer'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_manufacturers.plot(kind='barh', color='coral', edgecolor='black')
plt.title('Top 10 Manufacturers')
plt.xlabel('Number of Products')
plt.ylabel('Manufacturer')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 3. Product Type Distribution (Pie Chart)
product_type_counts = df['Product Type'].value_counts()

plt.figure(figsize=(8, 8))
product_type_counts.head(10).plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Product Type Distribution (Top 10)')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 4. GHX Commodity Type Distribution
commodity_counts = df['GHX Commodity Type'].value_counts()

plt.figure(figsize=(10, 6))
commodity_counts.plot(kind='bar', color='seagreen', edgecolor='black')
plt.title('GHX Commodity Type Distribution')
plt.xlabel('Commodity Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Top 10 Product Nouns
top_products = df['Product Noun'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_products.values, y=top_products.index, palette='viridis')
plt.title('Top 10 Product Nouns')
plt.xlabel('Count')
plt.ylabel('Product Noun')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Missing Values Heatmap
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100

missing_df = pd.DataFrame({'Missing Count': missing_data, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

print("\nMissing Values Summary:")
print(missing_df)

In [ ]:
# 7. UNSPSC Commodity Distribution
unspsc_counts = df['UNSPSC Commodity'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=unspsc_counts.values, y=unspsc_counts.index, palette='coolwarm')
plt.title('Top 10 UNSPSC Commodities')
plt.xlabel('Count')
plt.ylabel('UNSPSC Commodity')
plt.tight_layout()
plt.show()

In [ ]:
# 8. Order Qty by GHX Commodity Type
plt.figure(figsize=(12, 6))
commodity_qty = df.groupby('GHX Commodity Type')['Order Qty'].sum().sort_values(ascending=False)
commodity_qty.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Total Order Qty by Commodity Type')
plt.xlabel('Commodity Type')
plt.ylabel('Total Order Qty')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 9. Top MTFs (Medical Treatment Facilities) by Order Volume
mtf_orders = df.groupby('MTF Name')['Order Qty'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
mtf_orders.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Top 10 MTFs by Order Quantity')
plt.xlabel('Total Order Qty')
plt.ylabel('MTF Name')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 10. Product Size Distribution (where available)
size_data = df['Product Size'].dropna()

plt.figure(figsize=(10, 6))
size_data.value_counts().head(15).plot(kind='bar', color='darkorange', edgecolor='black')
plt.title('Product Size Distribution (Top 15)')
plt.xlabel('Product Size')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n✅ EDA Visualizations Complete!")

In [ ]:
# Basic profile
display(df.info())
display(df.describe(include='all').T)
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

In [ ]:
# Clean likely numeric column
if 'Order Qty' in df.columns:
    df['Order Qty'] = (
        df['Order Qty']
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('$', '', regex=False)
    )
    df['Order Qty'] = pd.to_numeric(df['Order Qty'], errors='coerce')

display(df[['Order Qty']].describe())

In [ ]:
# Quick EDA plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Order Qty' in df.columns:
    sns.histplot(df['Order Qty'].dropna(), bins=30, kde=True, ax=axes[0])
    axes[0].set_title('Order Qty Distribution')
else:
    axes[0].text(0.5, 0.5, 'Order Qty column missing', ha='center')

top_col = 'Manufacturer' if 'Manufacturer' in df.columns else df.columns[0]
top_vals = df[top_col].astype(str).value_counts().head(10)
sns.barplot(x=top_vals.values, y=top_vals.index, ax=axes[1])
axes[1].set_title(f'Top 10 in {top_col}')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Baseline ML
TARGET_COL = 'Order Qty'  # change this if needed

if TARGET_COL not in df.columns:
    raise ValueError(f'Target column {TARGET_COL} not found in dataframe.')

model_df = df.copy()
model_df = model_df.dropna(subset=[TARGET_COL])

X = model_df.drop(columns=[TARGET_COL])
y = model_df[TARGET_COL]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

is_regression = pd.api.types.is_numeric_dtype(y) and y.nunique() > 20

if is_regression:
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
else:
    y = y.astype(str)
    model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

clf = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf.fit(X_train, y_train)
preds = clf.predict(X_test)

if is_regression:
    print('Task: Regression')
    print('MAE :', round(mean_absolute_error(y_test, preds), 4))
    print('RMSE:', round(root_mean_squared_error(y_test, preds), 4))
else:
    print('Task: Classification')
    print('Accuracy:', round(accuracy_score(y_test, preds), 4))
    print('F1 macro:', round(f1_score(y_test, preds, average='macro'), 4))